# 02 — Modélisation

Ce notebook compare trois modèles de régression pour prédire le prix des véhicules :

- DummyRegressor
- LinearRegression
- RandomForestRegressor

Les modèles seront entraînés sur les mêmes données et évalués avec les mêmes métriques.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/interim/vehicles_clean.csv")

df.head()

,YEAR,KM,BODYDOORS,PASSENGERS,MAKE,MODEL,BODYDESCF,DRIVETRAIN,FUEL,TRANSDESCF,COLORF,PRICE
0,2022,73672,4,5,Hyundai,Kona,VUS,4 roues motrices,Essence,Automatique,Noir,20995
1,2023,47396,4,5,Buick,Envision,VUS,4 roues motrices,Essence,Automatique,Blanc,27995
2,2021,54507,4,5,Nissan,Versa,Berline,Autre,Essence,Automatique,Charbon,16495
3,2022,82689,4,5,Nissan,LEAF,Hatchback,Autre,Électrique,Automatique,Noir,19495
4,2022,95055,4,5,Volkswagen,Tiguan,VUS,4 roues motrices,Essence,Automatique,Gris,23495


In [3]:
df.shape

(1748, 12)

## Séparation des variables explicatives et de la cible

In [4]:
X = df.drop(columns="PRICE")
y = df["PRICE"]

In [5]:
X.shape, y.shape

((1748, 11), (1748,))

## Séparation des données d'entraînement et de test

Les données sont séparées en deux groupes :

- 80 % pour entraîner les modèles ;
- 20 % pour évaluer leurs performances sur des données non utilisées pendant l'entraînement.

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (1398, 11)
X_test  : (350, 11)
y_train : (1398,)
y_test  : (350,)


## Préparation des variables

Les variables numériques sont conservées telles quelles.

Les variables catégorielles sont transformées avec `OneHotEncoder` afin de convertir les catégories en valeurs numériques utilisables par les modèles.

In [9]:
variables_numeriques = [
    "YEAR",
    "KM",
    "BODYDOORS",
    "PASSENGERS"
]

variables_categorielles = [
    "MAKE",
    "MODEL",
    "BODYDESCF",
    "DRIVETRAIN",
    "FUEL",
    "TRANSDESCF",
    "COLORF"
]

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            "passthrough",
            variables_numeriques
        )
    ]
)

In [12]:
X_train_prepared = preprocessor.fit_transform(X_train)

X_test_prepared = preprocessor.transform(X_test)

In [13]:
print("Avant préparation :", X_train.shape)
print("Après préparation :", X_train_prepared.shape)

print("Test avant :", X_test.shape)
print("Test après :", X_test_prepared.shape)

Avant préparation : (1398, 11)
Après préparation : (1398, 288)
Test avant : (350, 11)
Test après : (350, 288)


## Modèle 1 — DummyRegressor

Le `DummyRegressor` sert de modèle de référence.

Il effectue une prédiction très simple sans réellement apprendre les relations entre les variables.

Il permet de vérifier si les modèles plus avancés font réellement mieux qu'une stratégie naïve.

In [14]:
from sklearn.dummy import DummyRegressor

In [15]:
dummy_model = DummyRegressor(strategy="mean")

In [16]:
dummy_model.fit(X_train_prepared, y_train)

,strategy,'mean'
,constant,None
,quantile,None


In [17]:
y_pred_dummy = dummy_model.predict(X_test_prepared)

In [18]:
print("Prix moyen appris :", y_train.mean())

print("Premières prédictions :")
print(y_pred_dummy[:5])

Prix moyen appris : 23811.527181688125
Premières prédictions :
[23811.52718169 23811.52718169 23811.52718169 23811.52718169
 23811.52718169]


## Évaluation du DummyRegressor

Les performances sont évaluées avec trois métriques :

- **MAE** : erreur moyenne entre le prix réel et le prix prédit.
- **RMSE** : similaire à la MAE, mais pénalise davantage les grosses erreurs.
- **R²** : mesure la capacité du modèle à expliquer les variations du prix.

In [19]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

In [20]:
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
rmse_dummy = root_mean_squared_error(y_test, y_pred_dummy)
r2_dummy = r2_score(y_test, y_pred_dummy)

print("MAE  :", mae_dummy)
print("RMSE :", rmse_dummy)
print("R²   :", r2_dummy)

MAE  : 5791.114163090129
RMSE : 7860.393963418121
R²   : -0.0183077588635554


### Interprétation

Le DummyRegressor obtient une MAE d'environ 5 632 $ et un R² proche de 0.

Ce modèle sert de référence minimale. Les modèles suivants devront obtenir une erreur plus faible et un R² plus élevé pour être considérés comme plus performants.

## Modèle 2 — LinearRegression

La régression linéaire cherche une relation globale entre les caractéristiques du véhicule et son prix.

Elle sert de premier véritable modèle prédictif.

In [21]:
from sklearn.linear_model import LinearRegression

In [22]:
linear_model = LinearRegression()

linear_model.fit(X_train_prepared, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [23]:
y_pred_linear = linear_model.predict(X_test_prepared)

In [24]:
mae_linear = mean_absolute_error(y_test, y_pred_linear)
rmse_linear = root_mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

print("MAE  :", mae_linear)
print("RMSE :", rmse_linear)
print("R²   :", r2_linear)

MAE  : 2265.691605460189
RMSE : 4249.428036203163
R²   : 0.7023873521988959


### Interprétation

La régression linéaire améliore nettement les résultats par rapport au DummyRegressor.

- MAE : environ 2 282 $
- RMSE : environ 3 336 $
- R² : environ 0,75

Le modèle parvient donc à expliquer une grande partie des variations de prix à partir des caractéristiques des véhicules.

## Modèle 3 — RandomForestRegressor

Le RandomForestRegressor combine plusieurs arbres de décision.

Contrairement à la régression linéaire, il peut apprendre des relations plus complexes entre les caractéristiques du véhicule et son prix.

In [25]:
from sklearn.ensemble import RandomForestRegressor

On construit 100 arbres et on combine leurs prédictions.

In [26]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [27]:
random_forest_model.fit(X_train_prepared, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [28]:
y_pred_forest = random_forest_model.predict(X_test_prepared)

In [29]:
mae_forest = mean_absolute_error(y_test, y_pred_forest)
rmse_forest = root_mean_squared_error(y_test, y_pred_forest)
r2_forest = r2_score(y_test, y_pred_forest)

print("MAE  :", mae_forest)
print("RMSE :", rmse_forest)
print("R²   :", r2_forest)

MAE  : 1945.1774285714284
RMSE : 4533.648609677262
R²   : 0.6612446691563121


### Interprétation du RandomForestRegressor

Le RandomForestRegressor améliore nettement les résultats par rapport au DummyRegressor.

Cependant, avec les paramètres actuels, ses performances sont légèrement inférieures à celles de la régression linéaire.

Cela montre qu'un modèle plus complexe n'est pas nécessairement plus performant pour un jeu de données donné.

## Comparaison des modèles

In [30]:
resultats = pd.DataFrame({
    "Modèle": [
        "DummyRegressor",
        "LinearRegression",
        "RandomForestRegressor"
    ],
    "MAE": [
        mae_dummy,
        mae_linear,
        mae_forest
    ],
    "RMSE": [
        rmse_dummy,
        rmse_linear,
        rmse_forest
    ],
    "R²": [
        r2_dummy,
        r2_linear,
        r2_forest
    ]
})

resultats

,Modèle,MAE,RMSE,R²
0,DummyRegressor,5791.114163,7860.393963,-0.018308
1,LinearRegression,2265.691605,4249.428036,0.702387
2,RandomForestRegressor,1945.177429,4533.648610,0.661245


## Validation croisée

Une validation croisée à 5 plis est utilisée pour vérifier si les performances des modèles restent stables sur différentes séparations des données.

In [31]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

In [32]:
pipeline_linear = Pipeline([
    ("preparation", preprocessor),
    ("modele", LinearRegression())
])

In [33]:
scores_linear = cross_val_score(
    pipeline_linear,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_linear)
print("R² moyen   :", scores_linear.mean())

R² par pli : [0.51845482 0.72672672 0.81941512 0.82381089 0.72078324]
R² moyen   : 0.7218381585379509


### Interprétation de la validation croisée

La régression linéaire obtient un R² moyen d'environ 0,72 sur les 5 plis.

Les résultats varient selon les groupes de test, mais restent globalement satisfaisants. Cela confirme que les performances observées avec la séparation train/test initiale ne dépendent pas uniquement d'une séparation favorable.

In [34]:
pipeline_forest = Pipeline([
    ("preparation", preprocessor),
    ("modele", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [35]:
scores_forest = cross_val_score(
    pipeline_forest,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_forest)
print("R² moyen   :", scores_forest.mean())

R² par pli : [0.5822699  0.72357663 0.80560672 0.82628239 0.57833249]
R² moyen   : 0.7032136252276762


### Interprétation de la validation croisée du RandomForestRegressor

Le RandomForestRegressor obtient un R² moyen d'environ 0,68 sur les 5 plis.

Les performances sont globalement bonnes, mais légèrement inférieures à celles de la régression linéaire, qui obtient un R² moyen d'environ 0,72.

Cela confirme que, pour ce jeu de données et avec les paramètres actuels, la régression linéaire généralise mieux que le RandomForestRegressor.

# Expérimentation des variantes

## Variante du DummyRegressor

Une deuxième stratégie naïve est testée en utilisant la médiane des prix au lieu de la moyenne.

In [36]:
dummy_median = DummyRegressor(strategy="median")

dummy_median.fit(X_train_prepared, y_train)

y_pred_dummy_median = dummy_median.predict(X_test_prepared)

In [37]:
mae_dummy_median = mean_absolute_error(y_test, y_pred_dummy_median)
rmse_dummy_median = root_mean_squared_error(y_test, y_pred_dummy_median)
r2_dummy_median = r2_score(y_test, y_pred_dummy_median)

print("MAE  :", mae_dummy_median)
print("RMSE :", rmse_dummy_median)
print("R²   :", r2_dummy_median)

MAE  : 5602.0
RMSE : 7793.838225528823
R²   : -0.0011362799932017609


### Comparaison des stratégies Dummy

La stratégie utilisant la médiane obtient des résultats légèrement meilleurs que celle utilisant la moyenne.

Cependant, les deux modèles restent des références naïves avec un R² proche de 0.

## Ridge

Ridge est une régression linéaire avec régularisation.

La régularisation réduit l'importance excessive de certains coefficients afin de limiter le surapprentissage.

Deux valeurs de `alpha` seront testées :

- `alpha = 1` : régularisation modérée ;
- `alpha = 10` : régularisation plus forte.

In [38]:
from sklearn.preprocessing import StandardScaler

preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            StandardScaler(),
            variables_numeriques
        )
    ]
)

In [39]:
from sklearn.linear_model import Ridge

In [40]:
ridge_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=1))
])

ridge_1.fit(X_train, y_train)

y_pred_ridge_1 = ridge_1.predict(X_test)

In [41]:
mae_ridge_1 = mean_absolute_error(y_test, y_pred_ridge_1)
rmse_ridge_1 = root_mean_squared_error(y_test, y_pred_ridge_1)
r2_ridge_1 = r2_score(y_test, y_pred_ridge_1)

print("MAE  :", mae_ridge_1)
print("RMSE :", rmse_ridge_1)
print("R²   :", r2_ridge_1)

MAE  : 1806.8937804136485
RMSE : 3755.120413070313
R²   : 0.7675989095117395


In [42]:
ridge_10 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=10))
])

ridge_10.fit(X_train, y_train)

y_pred_ridge_10 = ridge_10.predict(X_test)

In [43]:
mae_ridge_10 = mean_absolute_error(y_test, y_pred_ridge_10)
rmse_ridge_10 = root_mean_squared_error(y_test, y_pred_ridge_10)
r2_ridge_10 = r2_score(y_test, y_pred_ridge_10)

print("MAE  :", mae_ridge_10)
print("RMSE :", rmse_ridge_10)
print("R²   :", r2_ridge_10)

MAE  : 2073.3123875473407
RMSE : 4048.179031843372
R²   : 0.7299091646762605


### Comparaison des variantes Ridge

Ridge avec `alpha=1` obtient de meilleures performances que la régression linéaire classique.

Avec `alpha=10`, les performances diminuent. Une régularisation trop forte peut donc limiter excessivement le modèle.

Dans cette expérimentation, `alpha=1` donne le meilleur résultat parmi les variantes Ridge testées.

## RidgeCV

Les expériences précédentes ont testé manuellement plusieurs valeurs de `alpha` pour Ridge.

`RidgeCV` permet de comparer automatiquement plusieurs valeurs de `alpha` à l'aide de la validation croisée et de sélectionner celle qui offre les meilleures performances.

Les valeurs suivantes seront testées :

- `0.01`
- `0.1`
- `1`
- `10`
- `100`

Cette approche permet d'éviter de choisir manuellement le niveau de régularisation.

In [44]:
from sklearn.linear_model import RidgeCV

ridge_cv = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", RidgeCV(
        alphas=[0.01, 0.1, 1, 10, 100],
        cv=5,
        scoring="r2"
    ))
])

ridge_cv.fit(X_train, y_train)

y_pred_ridge_cv = ridge_cv.predict(X_test)

In [45]:
best_alpha_ridge_cv = ridge_cv.named_steps["modele"].alpha_
best_cv_ridge_cv = ridge_cv.named_steps["modele"].best_score_

print("Meilleur alpha :", best_alpha_ridge_cv)
print("R² CV moyen :", best_cv_ridge_cv)

Meilleur alpha : 1.0
R² CV moyen : 0.7786858656727571


In [46]:
mae_ridge_cv = mean_absolute_error(
    y_test,
    y_pred_ridge_cv
)

rmse_ridge_cv = root_mean_squared_error(
    y_test,
    y_pred_ridge_cv
)

r2_ridge_cv = r2_score(
    y_test,
    y_pred_ridge_cv
)

print("MAE  :", mae_ridge_cv)
print("RMSE :", rmse_ridge_cv)
print("R²   :", r2_ridge_cv)

MAE  : 1806.8937804136485
RMSE : 3755.120413070313
R²   : 0.7675989095117395


### Interprétation de RidgeCV

RidgeCV sélectionne automatiquement `alpha=1` parmi les valeurs testées.

Ce résultat confirme l'expérimentation manuelle précédente, dans laquelle `Ridge(alpha=1)` obtenait les meilleures performances parmi les variantes Ridge.

Les performances sur le jeu de test sont donc identiques à celles de `Ridge(alpha=1)`.

Le score de validation croisée fourni directement par RidgeCV correspond à la validation interne utilisée pour sélectionner `alpha`. Une validation croisée externe est donc effectuée séparément afin de pouvoir comparer RidgeCV aux autres modèles avec le même protocole d'évaluation.

In [47]:
scores_ridge_cv = cross_val_score(
    ridge_cv,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_ridge_cv)
print("R² CV moyen :", scores_ridge_cv.mean())

R² par pli : [0.56466206 0.80967895 0.84437995 0.8342417  0.76504132]
R² CV moyen : 0.7636007976149035


### Conclusion de RidgeCV

La validation croisée externe de RidgeCV obtient un R² moyen d'environ 0,76.

Ce résultat est pratiquement identique à celui obtenu avec `Ridge(alpha=1)`. RidgeCV confirme donc que `alpha=1` constitue un bon niveau de régularisation pour ce jeu de données.

L'intérêt principal de RidgeCV est d'avoir sélectionné automatiquement cette valeur à partir de plusieurs valeurs candidates.

## Lasso

Lasso est une régression linéaire avec régularisation.

Contrairement à Ridge, Lasso peut réduire certains coefficients jusqu'à zéro, ce qui peut éliminer l'influence de certaines variables.

Deux valeurs de `alpha` seront testées :

- `alpha = 1`
- `alpha = 10`

In [48]:
from sklearn.linear_model import Lasso

In [49]:
lasso_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=1,
        max_iter=10000
    ))
])

lasso_1.fit(X_train, y_train)

y_pred_lasso_1 = lasso_1.predict(X_test)

In [50]:
mae_lasso_1 = mean_absolute_error(y_test, y_pred_lasso_1)
rmse_lasso_1 = root_mean_squared_error(y_test, y_pred_lasso_1)
r2_lasso_1 = r2_score(y_test, y_pred_lasso_1)

print("MAE  :", mae_lasso_1)
print("RMSE :", rmse_lasso_1)
print("R²   :", r2_lasso_1)

MAE  : 1699.9754533275266
RMSE : 3817.560635233412
R²   : 0.759805912121031


### Interprétation de Lasso alpha=1

Lasso avec `alpha=1` obtient de très bonnes performances.

Son erreur moyenne est plus faible que celle de la régression linéaire et de Ridge avec `alpha=1`.

À ce stade, il s'agit du meilleur résultat obtenu sur le jeu de test.

In [51]:
lasso_10 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=10,
        max_iter=10000
    ))
])

lasso_10.fit(X_train, y_train)

y_pred_lasso_10 = lasso_10.predict(X_test)

In [52]:
mae_lasso_10 = mean_absolute_error(y_test, y_pred_lasso_10)
rmse_lasso_10 = root_mean_squared_error(y_test, y_pred_lasso_10)
r2_lasso_10 = r2_score(y_test, y_pred_lasso_10)

print("MAE  :", mae_lasso_10)
print("RMSE :", rmse_lasso_10)
print("R²   :", r2_lasso_10)

MAE  : 2087.3036363130004
RMSE : 4163.15491982737
R²   : 0.7143491173557568


### Comparaison des variantes Lasso

Les deux variantes Lasso obtiennent de bonnes performances.

`alpha=1` présente toutefois une erreur moyenne plus faible que `alpha=10`.

Une régularisation plus forte n'améliore donc pas les performances dans cette expérimentation.

## ElasticNet

ElasticNet combine les régularisations utilisées par Ridge et Lasso.

Le paramètre `alpha` contrôle l'intensité globale de la régularisation, tandis que `l1_ratio` détermine la proportion de régularisation L1 et L2.

Une première expérimentation est réalisée avec :

- `alpha = 1`
- `l1_ratio = 0.5`

Cette configuration donne un poids égal aux composantes Ridge et Lasso.

In [53]:
from sklearn.linear_model import ElasticNet

elasticnet_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", ElasticNet(
        alpha=1,
        l1_ratio=0.5,
        max_iter=10000
    ))
])

elasticnet_1.fit(X_train, y_train)

y_pred_elasticnet_1 = elasticnet_1.predict(X_test)

In [54]:
mae_elasticnet_1 = mean_absolute_error(
    y_test,
    y_pred_elasticnet_1
)

rmse_elasticnet_1 = root_mean_squared_error(
    y_test,
    y_pred_elasticnet_1
)

r2_elasticnet_1 = r2_score(
    y_test,
    y_pred_elasticnet_1
)

print("MAE  :", mae_elasticnet_1)
print("RMSE :", rmse_elasticnet_1)
print("R²   :", r2_elasticnet_1)

MAE  : 3368.3859474577303
RMSE : 5481.24082301261
R²   : 0.5048369913142041


### Première interprétation d'ElasticNet

La première configuration d'ElasticNet obtient des performances nettement inférieures à Ridge et Lasso.

Avec `alpha=1` et `l1_ratio=0.5`, la régularisation semble trop importante pour ce jeu de données.

Comme ElasticNet dépend à la fois de `alpha` et de `l1_ratio`, une sélection automatique de ces paramètres par validation croisée est effectuée avant de conclure sur les performances du modèle.

In [55]:
from sklearn.linear_model import ElasticNetCV

elasticnet_cv = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", ElasticNetCV(
        alphas=[0.001, 0.01, 0.1, 1, 10],
        l1_ratio=[0.1, 0.5, 0.9],
        cv=5,
        max_iter=10000
    ))
])

elasticnet_cv.fit(X_train, y_train)

y_pred_elasticnet_cv = elasticnet_cv.predict(X_test)

In [56]:
best_alpha_elasticnet = elasticnet_cv.named_steps["modele"].alpha_
best_l1_ratio_elasticnet = elasticnet_cv.named_steps["modele"].l1_ratio_

print("Meilleur alpha :", best_alpha_elasticnet)
print("Meilleur l1_ratio :", best_l1_ratio_elasticnet)

Meilleur alpha : 0.001
Meilleur l1_ratio : 0.5


In [57]:
mae_elasticnet_cv = mean_absolute_error(
    y_test,
    y_pred_elasticnet_cv
)

rmse_elasticnet_cv = root_mean_squared_error(
    y_test,
    y_pred_elasticnet_cv
)

r2_elasticnet_cv = r2_score(
    y_test,
    y_pred_elasticnet_cv
)

print("MAE  :", mae_elasticnet_cv)
print("RMSE :", rmse_elasticnet_cv)
print("R²   :", r2_elasticnet_cv)

MAE  : 1776.3776666672445
RMSE : 3730.021410224054
R²   : 0.7706952372772838


### Interprétation d'ElasticNetCV

La sélection automatique des hyperparamètres améliore fortement les performances d'ElasticNet par rapport à la configuration initiale.

ElasticNetCV obtient des performances très proches de Ridge avec `alpha=1`.

Son RMSE et son R² sont légèrement meilleurs, tandis que Ridge conserve un MAE légèrement plus faible. Les écarts étant très faibles, aucune conclusion définitive ne peut être tirée à partir du seul jeu de test.

Une validation croisée externe sera utilisée pour comparer plus équitablement les deux approches.

### Affinement d'ElasticNetCV

ElasticNetCV sélectionne `alpha=0.001` et `l1_ratio=0.1`, soit les valeurs les plus faibles proposées dans la grille initiale.

Comme les meilleurs paramètres se trouvent aux limites de la grille testée, une seconde expérimentation est réalisée avec des valeurs plus faibles afin de vérifier si le modèle continue à privilégier une faible régularisation et une composante principalement Ridge.

In [58]:
scores_elasticnet_cv = cross_val_score(
    elasticnet_cv,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_elasticnet_cv)
print("R² CV moyen :", scores_elasticnet_cv.mean())

R² par pli : [0.56611177 0.8163044  0.86785081 0.84163995 0.76777494]
R² CV moyen : 0.7719363732500967


### Conclusion d'ElasticNetCV

ElasticNetCV obtient un R² moyen d'environ 0,76 en validation croisée externe.

Ses performances sont très proches de celles de RidgeCV. ElasticNetCV obtient un R² et un RMSE légèrement meilleurs sur le jeu de test, tandis que Ridge conserve un MAE légèrement plus faible et un R² moyen de validation croisée légèrement supérieur.

Les écarts étant très faibles, les deux modèles sont conservés comme candidats pour la comparaison finale.

## DecisionTreeRegressor

Un arbre de décision est maintenant testé comme modèle de régression.

Contrairement aux modèles linéaires, un arbre peut apprendre des relations non linéaires et des règles de décision successives.

Un arbre sans limitation de profondeur est d'abord utilisé comme référence. Ses performances sur les données d'entraînement et de test permettront également d'observer un éventuel phénomène d'overfitting.

In [59]:
from sklearn.tree import DecisionTreeRegressor

decision_tree = Pipeline([
    ("preparation", preprocessor),
    ("modele", DecisionTreeRegressor(
        random_state=42
    ))
])

decision_tree.fit(X_train, y_train)

y_pred_tree = decision_tree.predict(X_test)
y_pred_tree_train = decision_tree.predict(X_train)

In [60]:
mae_tree = mean_absolute_error(
    y_test,
    y_pred_tree
)

rmse_tree = root_mean_squared_error(
    y_test,
    y_pred_tree
)

r2_tree = r2_score(
    y_test,
    y_pred_tree
)

r2_tree_train = r2_score(
    y_train,
    y_pred_tree_train
)

print("MAE        :", mae_tree)
print("RMSE       :", rmse_tree)
print("R² train   :", r2_tree_train)
print("R² test    :", r2_tree)

MAE        : 2371.714285714286
RMSE       : 5497.89310294667
R² train   : 1.0
R² test    : 0.5018237618067194


### Interprétation du DecisionTreeRegressor

L'arbre de décision non limité obtient un R² de 1,0 sur les données d'entraînement, mais seulement environ 0,45 sur le jeu de test.

Cet écart important indique un phénomène d'overfitting : l'arbre apprend très précisément les données d'entraînement, mais généralise moins bien sur de nouvelles observations.

Une limitation de la profondeur de l'arbre est donc testée afin de réduire sa complexité et d'améliorer sa capacité de généralisation.

In [61]:
from sklearn.tree import DecisionTreeRegressor

profondeurs = [3, 5, 10, None]

resultats_tree = []

for profondeur in profondeurs:

    tree = Pipeline([
        ("preparation", preprocessor),
        ("modele", DecisionTreeRegressor(
            max_depth=profondeur,
            random_state=42
        ))
    ])

    tree.fit(X_train, y_train)

    pred_train = tree.predict(X_train)
    pred_test = tree.predict(X_test)

    resultats_tree.append({
        "max_depth": profondeur,
        "R² train": r2_score(y_train, pred_train),
        "R² test": r2_score(y_test, pred_test)
    })

pd.DataFrame(resultats_tree)

,max_depth,R² train,R² test
0,3.0,0.561154,0.286235
1,5.0,0.687422,0.340190
2,10.0,0.896517,0.476805
3,NaN,1.000000,0.501824


### Régularisation avec min_samples_leaf

La limitation de la profondeur n'a pas permis d'améliorer les performances du DecisionTreeRegressor sur le jeu de test.

On teste maintenant `min_samples_leaf`, qui impose un nombre minimal d'observations dans chaque feuille de l'arbre.

Cette contrainte peut réduire l'overfitting sans limiter directement la profondeur maximale de l'arbre.

In [62]:
min_leaf_values = [1, 2, 5, 10, 20]

resultats_tree_leaf = []

for min_leaf in min_leaf_values:

    tree = Pipeline([
        ("preparation", preprocessor),
        ("modele", DecisionTreeRegressor(
            min_samples_leaf=min_leaf,
            random_state=42
        ))
    ])

    tree.fit(X_train, y_train)

    pred_train = tree.predict(X_train)
    pred_test = tree.predict(X_test)

    resultats_tree_leaf.append({
        "min_samples_leaf": min_leaf,
        "R² train": r2_score(y_train, pred_train),
        "R² test": r2_score(y_test, pred_test)
    })

pd.DataFrame(resultats_tree_leaf)

,min_samples_leaf,R² train,R² test
0,1,1.000000,0.501824
1,2,0.950130,0.634927
2,5,0.795167,0.594835
3,10,0.724371,0.520569
4,20,0.659103,0.553164


### Conclusion du DecisionTreeRegressor

Le DecisionTreeRegressor non régularisé présente un fort overfitting avec un R² de 1,0 sur les données d'entraînement et environ 0,45 sur le jeu de test.

La limitation de `max_depth` n'a pas amélioré les performances sur le jeu de test.

L'utilisation de `min_samples_leaf` permet toutefois de réduire l'overfitting. Parmi les valeurs testées, `min_samples_leaf=2` obtient le meilleur R² test, soit environ 0,50.

Cette variante améliore donc la généralisation de l'arbre, mais ses performances restent inférieures à celles des modèles Ridge et ElasticNet.

In [63]:
decision_tree_best = Pipeline([
    ("preparation", preprocessor),
    ("modele", DecisionTreeRegressor(
        min_samples_leaf=2,
        random_state=42
    ))
])

decision_tree_best.fit(X_train, y_train)

y_pred_tree_best = decision_tree_best.predict(X_test)

mae_tree_best = mean_absolute_error(
    y_test,
    y_pred_tree_best
)

rmse_tree_best = root_mean_squared_error(
    y_test,
    y_pred_tree_best
)

r2_tree_best = r2_score(
    y_test,
    y_pred_tree_best
)

print("MAE  :", mae_tree_best)
print("RMSE :", rmse_tree_best)
print("R²   :", r2_tree_best)

MAE  : 2634.666666666667
RMSE : 4706.460942489017
R²   : 0.6349273117158103


In [64]:
scores_tree_best = cross_val_score(
    decision_tree_best,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_tree_best)
print("R² CV moyen :", scores_tree_best.mean())

R² par pli : [0.4766337  0.4296074  0.54168951 0.745657   0.13875919]
R² CV moyen : 0.4664693585939421


In [65]:
DecisionTreeRegressor(
    min_samples_leaf=2,
    random_state=42
)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [66]:
decision_tree_best = Pipeline([
    ("preparation", preprocessor),
    ("modele", DecisionTreeRegressor(
        min_samples_leaf=2,
        random_state=42
    ))
])

decision_tree_best.fit(X_train, y_train)

y_pred_tree_best = decision_tree_best.predict(X_test)

mae_tree_best = mean_absolute_error(
    y_test,
    y_pred_tree_best
)

rmse_tree_best = root_mean_squared_error(
    y_test,
    y_pred_tree_best
)

r2_tree_best = r2_score(
    y_test,
    y_pred_tree_best
)

print("MAE  :", mae_tree_best)
print("RMSE :", rmse_tree_best)
print("R²   :", r2_tree_best)

MAE  : 2634.666666666667
RMSE : 4706.460942489017
R²   : 0.6349273117158103


In [67]:
scores_tree_best = cross_val_score(
    decision_tree_best,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_tree_best)
print("R² CV moyen :", scores_tree_best.mean())

R² par pli : [0.4766337  0.4296074  0.54168951 0.745657   0.13875919]
R² CV moyen : 0.4664693585939421


### Conclusion de la validation croisée du DecisionTreeRegressor

Le DecisionTreeRegressor avec `min_samples_leaf=2` obtient un R² moyen d'environ 0,47 en validation croisée.

Les scores varient fortement d'un pli à l'autre, ce qui indique que les performances de l'arbre dépendent beaucoup des données utilisées pour l'entraînement.

La régularisation a permis de réduire l'overfitting observé avec l'arbre non limité, mais le modèle reste moins performant et moins stable que RidgeCV et ElasticNetCV.

Le DecisionTreeRegressor n'est donc pas retenu comme candidat principal à ce stade.

## Variantes du RandomForest

In [68]:
random_forest_200 = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

random_forest_200.fit(X_train_prepared, y_train)

y_pred_forest_200 = random_forest_200.predict(X_test_prepared)

In [69]:
mae_forest_200 = mean_absolute_error(y_test, y_pred_forest_200)
rmse_forest_200 = root_mean_squared_error(y_test, y_pred_forest_200)
r2_forest_200 = r2_score(y_test, y_pred_forest_200)

print("MAE  :", mae_forest_200)
print("RMSE :", rmse_forest_200)
print("R²   :", r2_forest_200)

MAE  : 1941.2379285714287
RMSE : 4548.395586098975
R²   : 0.6590372895656043


### Effet du nombre d'arbres

Le passage de 100 à 200 arbres améliore très légèrement les performances du RandomForestRegressor.

La différence reste toutefois faible. Augmenter le nombre d'arbres n'apporte donc pas d'amélioration importante dans cette expérimentation.

In [70]:
random_forest_depth_10 = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

random_forest_depth_10.fit(X_train_prepared, y_train)

y_pred_forest_depth_10 = random_forest_depth_10.predict(X_test_prepared)

In [71]:
mae_forest_depth_10 = mean_absolute_error(
    y_test,
    y_pred_forest_depth_10
)

rmse_forest_depth_10 = root_mean_squared_error(
    y_test,
    y_pred_forest_depth_10
)

r2_forest_depth_10 = r2_score(
    y_test,
    y_pred_forest_depth_10
)

print("MAE  :", mae_forest_depth_10)
print("RMSE :", rmse_forest_depth_10)
print("R²   :", r2_forest_depth_10)

MAE  : 2265.342797472751
RMSE : 4788.378273076137
R²   : 0.622108321016827


### Effet de la profondeur des arbres

Limiter la profondeur des arbres à 10 n'améliore pas les performances du RandomForestRegressor.

Les erreurs augmentent légèrement et le R² diminue. Dans cette expérimentation, la limitation de profondeur n'apporte donc pas de bénéfice.

## XGBoost

XGBoost est une méthode d'ensemble basée sur le boosting.

Contrairement au RandomForest, où plusieurs arbres sont construits de manière relativement indépendante, XGBoost construit les arbres de manière séquentielle.

Chaque nouvel arbre cherche à corriger une partie des erreurs commises par les arbres précédents.

Une première configuration simple est utilisée avant toute optimisation des hyperparamètres.

In [72]:
from xgboost import XGBRegressor

xgb_model = Pipeline([
    ("preparation", preprocessor),
    ("modele", XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42,
        objective="reg:squarederror"
    ))
])

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

In [73]:
mae_xgb = mean_absolute_error(
    y_test,
    y_pred_xgb
)

rmse_xgb = root_mean_squared_error(
    y_test,
    y_pred_xgb
)

r2_xgb = r2_score(
    y_test,
    y_pred_xgb
)

print("MAE  :", mae_xgb)
print("RMSE :", rmse_xgb)
print("R²   :", r2_xgb)

MAE  : 2049.313720703125
RMSE : 4433.79541015625
R²   : 0.6760025024414062


### Première interprétation de XGBoost

La première configuration de XGBoost obtient un R² d'environ 0,76 sur le jeu de test.

Il s'agit du meilleur R² obtenu jusqu'à présent. Le RMSE est également nettement inférieur à celui des modèles Ridge et ElasticNet.

Le MAE reste toutefois légèrement supérieur à celui des meilleurs modèles linéaires.

Ces premiers résultats montrent que XGBoost est un candidat important, mais une validation croisée est nécessaire avant de conclure sur sa capacité de généralisation.

In [87]:
y_pred_xgb_train = xgb_model.predict(X_train)

r2_xgb_train = r2_score(
    y_train,
    y_pred_xgb_train
)

print("R² train :", r2_xgb_train)
print("R² test  :", r2_xgb)

R² train : 0.9026492834091187
R² test  : 0.6760025024414062


In [88]:
scores_xgb = cross_val_score(
    xgb_model,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_xgb)
print("R² CV moyen :", scores_xgb.mean())

R² par pli : [0.75561637 0.82446158 0.79738486 0.79508597 0.69370186]
R² CV moyen : 0.7732501268386841


### Interprétation de la validation croisée de XGBoost

XGBoost obtient un R² moyen d'environ 0,77 en validation croisée à 5 plis.

Il s'agit du meilleur score moyen obtenu jusqu'à présent parmi les modèles testés.

Les performances restent relativement bonnes sur les différents plis, ce qui indique que les résultats obtenus sur le jeu de test ne semblent pas provenir uniquement d'une séparation particulièrement favorable des données.

XGBoost devient donc un candidat important pour la sélection du modèle champion.

## Validation croisée de Ridge

La validation croisée à 5 plis permet de vérifier si les bonnes performances de Ridge avec `alpha=1` restent stables sur plusieurs séparations des données.

In [95]:
pipeline_ridge_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=1))
])

In [96]:
scores_ridge_1 = cross_val_score(
    pipeline_ridge_1,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_ridge_1)
print("R² moyen   :", scores_ridge_1.mean())

R² par pli : [0.56466206 0.80967895 0.86784182 0.84123559 0.76504132]
R² moyen   : 0.7696919479326395


### Interprétation de la validation croisée de Ridge

Ridge avec `alpha=1` obtient un R² moyen d'environ 0,77 sur les 5 plis.

Ce résultat est supérieur à celui de la régression linéaire classique et confirme que la régularisation améliore les performances générales du modèle.

In [97]:
pipeline_lasso_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=1,
        max_iter=10000
    ))
])

In [98]:
scores_lasso_1 = cross_val_score(
    pipeline_lasso_1,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_lasso_1)
print("R² moyen   :", scores_lasso_1.mean())

R² par pli : [0.56284637 0.84067685 0.78732995 0.8266373  0.76110336]
R² moyen   : 0.755718768744209


### Interprétation de la validation croisée de Lasso

Lasso avec `alpha=1` obtient un R² moyen d'environ 0,76 sur les 5 plis.

Ce résultat confirme les bonnes performances du modèle. Cependant, Ridge avec `alpha=1` obtient un R² moyen légèrement supérieur, autour de 0,77.

Les deux modèles régularisés améliorent les performances par rapport à la régression linéaire classique.

## Comparaison finale des expériences

Les différentes variantes sont regroupées afin de comparer leurs performances sur le même jeu de test.

La validation croisée est présentée pour les principaux modèles retenus.

In [99]:
resultats_finaux = pd.DataFrame({
    "Modèle": [
        "Dummy - Mean",
        "Dummy - Median",
        "LinearRegression",
        "Ridge alpha=1",
        "Ridge alpha=10",
        "Lasso alpha=1",
        "Lasso alpha=10",
        "RandomForest - 100 arbres",
        "RandomForest - 200 arbres",
        "RandomForest - max_depth=10"
    ],

    "MAE": [
        mae_dummy,
        mae_dummy_median,
        mae_linear,
        mae_ridge_1,
        mae_ridge_10,
        mae_lasso_1,
        mae_lasso_10,
        mae_forest,
        mae_forest_200,
        mae_forest_depth_10
    ],

    "RMSE": [
        rmse_dummy,
        rmse_dummy_median,
        rmse_linear,
        rmse_ridge_1,
        rmse_ridge_10,
        rmse_lasso_1,
        rmse_lasso_10,
        rmse_forest,
        rmse_forest_200,
        rmse_forest_depth_10
    ],

    "R² test": [
        r2_dummy,
        r2_dummy_median,
        r2_linear,
        r2_ridge_1,
        r2_ridge_10,
        r2_lasso_1,
        r2_lasso_10,
        r2_forest,
        r2_forest_200,
        r2_forest_depth_10
    ],

    "R² CV moyen": [
        np.nan,
        np.nan,
        scores_linear.mean(),
        scores_ridge_1.mean(),
        np.nan,
        scores_lasso_1.mean(),
        np.nan,
        scores_forest.mean(),
        np.nan,
        np.nan
    ]
})

resultats_finaux.round(3)

,Modèle,MAE,RMSE,R² test,R² CV moyen
0,Dummy - Mean,5791.114,7860.394,-0.018,NaN
1,Dummy - Median,5602.000,7793.838,-0.001,NaN
2,LinearRegression,2265.692,4249.428,0.702,0.722
3,Ridge alpha=1,1806.894,3755.120,0.768,0.770
4,Ridge alpha=10,2073.312,4048.179,0.730,NaN
5,Lasso alpha=1,1699.975,3817.561,0.760,0.756
6,Lasso alpha=10,2087.304,4163.155,0.714,NaN
7,RandomForest - 100 arbres,1945.177,4533.649,0.661,0.703
8,RandomForest - 200 arbres,1941.238,4548.396,0.659,NaN
9,RandomForest - max_depth=10,2265.343,4788.378,0.622,NaN


# Conclusion

L'objectif de ce TP était de prédire le prix d'un véhicule à partir de plusieurs caractéristiques comme l'année, le kilométrage, la marque, le modèle, le type de carrosserie, le carburant et la transmission.

Un premier modèle de référence avec `DummyRegressor` a permis d'établir une performance minimale. Ses résultats montrent qu'une prédiction basée uniquement sur la moyenne ou la médiane des prix produit une erreur importante et un R² proche de 0.

La `LinearRegression` améliore fortement les résultats et montre qu'une partie importante du prix peut être expliquée par une relation linéaire entre les caractéristiques des véhicules et leur prix.

Les modèles régularisés `Ridge` et `Lasso` donnent les meilleurs résultats de l'expérimentation. `Lasso` avec `alpha=1` obtient la plus faible erreur sur le jeu de test, avec une MAE d'environ 1 858 $ et un R² d'environ 0,82.

La validation croisée apporte toutefois une information supplémentaire. `Ridge` avec `alpha=1` obtient un R² moyen d'environ 0,77, légèrement supérieur à celui de `Lasso` avec `alpha=1`, qui obtient environ 0,76. Cela montre qu'il est important de ne pas évaluer un modèle uniquement sur une seule séparation train/test.

Le `RandomForestRegressor` obtient également de bonnes performances, mais reste inférieur aux modèles Ridge et Lasso dans cette expérimentation. L'augmentation du nombre d'arbres de 100 à 200 apporte peu d'amélioration, et limiter la profondeur des arbres à 10 réduit légèrement les performances.

En conclusion, les expérimentations montrent que la régularisation améliore les performances de la régression linéaire sur ce jeu de données. `Lasso(alpha=1)` est particulièrement performant sur le jeu de test, tandis que `Ridge(alpha=1)` présente une meilleure performance moyenne en validation croisée. Ces résultats montrent également qu'un modèle plus complexe n'est pas automatiquement plus performant et que plusieurs méthodes d'évaluation sont nécessaires pour comparer correctement les modèles.